In [53]:
import os
import certifi
from dotenv import load_dotenv

# Gemini integration
from langchain_google_genai import ChatGoogleGenerativeAI

# Tavily search tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain_core.prompts import PromptTemplate

from langchain.tools import tool
import requests

load_dotenv()

True

In [38]:
from langchain.agents import create_react_agent, AgentExecutor

In [39]:
# ============================================
# LOAD ENV VARIABLES
# ============================================

os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY= os.getenv("WEATHERSTACK_API_KEY")

In [40]:
search_tool = TavilySearchResults(max_results=2)

In [60]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url, timeout=15)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data.get('error')}"

    return (
        f"Weather in {city}: "
        f"{data['current']['temperature']}°C, "
        f"{data['current']['weather_descriptions'][0]}. "
        f"Humidity: {data['current']['humidity']}%."
    )

In [10]:
result = search_tool.invoke("Give me the latest news on ai")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': '### Singapore partners with Anthropic on public sector AI deployments\n\n1182 view(s)\n\nSee all\n\n## Latest\n\nView All Latest\n\nCybersecurity AI\n\nOctober 23, 2026\n\n# OpenAI and Poland’s NASK partner on AI safety research\n\nAI in Action\n\nOctober 23, 2026\n\n# Microsoft AI diagnostic tool outperforms doctors in clinical tests\n\nGovernance, Regulation & Policy\n\nOctober 22, 2026\n\n# State legislatures become de facto AI regulators as Congress stalls\n\nInterviews\n\nOctober 20, 2026\n\n# Sebastian Schmidt, Digital Science: On grounding enterprise AI in scientific evidence\n\nGovernment & Public Sector AI\n\nOctober 20, 2026\n\n# Germany and NVIDIA sign industrial AI declaration in Berlin\n\nGovernance, Regulation & Policy\n\nOctober 20, 2026\n\n# Sarawak expands voice AI trials for government services [...] # Hershey applies AI across its supply chain operations\n\n# With higher risk and broader impact, AI

In [42]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    api_key=GOOGLE_API_KEY
)

In [54]:
#prompt
#prompt = hub.pull("hwchase17/react-multi-input-json")

template = """
You are a helpful AI assistant that can use tools.

You have access to these tools:

{tools}

You MUST follow this exact format:

Question: the user's question

Thought: explain what you need to do.

Action: one of [{tool_names}]
Action Input: the input for the selected tool

Observation: the result returned by the tool.

Repeat Thought, Action, Action Input, and Observation
if more tools are required.

When you have enough information, use:

Thought: I now know the final answer.
Final Answer: your final response.

STRICT RULES:
1. Never use JSON for actions.
2. Never use Markdown code blocks.
3. Action must contain only the exact tool name.
4. Action Input must contain the tool input.
5. Never combine Action and Action Input on one line.
6. If no tool is needed, directly provide Final Answer.
7. Never invent tool names.

Question: {input}

Thought:{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)

In [14]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [45]:
#tool
tools = [search_tool, get_weather_data]

In [55]:
#create agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [56]:
#executor

agent_executor= AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

In [62]:
#run
response = agent_executor.invoke({
    "input":(
        "Find the capital of India and find its current weather."
    )
})



> Entering new AgentExecutor chain...
I need to determine the capital of India (which is New Delhi) and check its current weather using the weather tool.

Action: get_weather_data
Action Input: New DelhiCould not fetch weather data for New Delhi: {'code': 101, 'type': 'invalid_access_key', 'info': 'You have not supplied a valid API Access Key. [Technical Support: support@apilayer.com]'}Thought: Since the weather tool encountered an API error, I will use Tavily search to find the current weather in New Delhi, the capital of India.

Action: tavily_search_results_json
Action Input: current weather in New Delhi[{'url': 'https://www.aqi.in/weather/us/india/delhi/new-delhi', 'content': "## FAQs About New Delhi Weather Conditions\n\n1. What is the current temperature in New Delhi today - 01 November 2026?\n\nThe current temperature in New Delhi is 24°C, with a feels-like temperature of 25°C. Today's forecast shows a high of 30°C and a low of 20°C, giving a 10 degree swing through the day as

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.
Please retry in 15h56m4.424354585s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  description: "Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis

KeyboardInterrupt: 

In [ ]:
print(response["output"])

{'input': 'Find the capital of India and find its current weather.', 'output': 'The capital of India is New Delhi. The current weather in New Delhi is approximately 24°C with hazy conditions and humidity around 59%.'}
